# CKD Counterfactual Demo (Clinically Constrained)

This notebook demonstrates *inference-only* counterfactual generation using the repo artifacts:

- A trained model (default: `models/rf_augmented_42_v1.joblib`)
- The fitted canonical preprocessor (`models/preprocessor.pkl`)

It uses the clinically constrained, gradient-free search implemented in `src/counterfactual.py`.

Key safeguards:

- **Auto-target**: if `target_class=None`, the notebook flips the model's current predicted label (avoids label-convention mismatch).
- **Standardized proximity** (`proximity_mode='zscore'`): distance is normalized by train-set feature std dev (reduces range bias).
- **Pareto selection** (`selection='pareto'`): returns non-dominated CFs over proximity/sparsity/p_target/robustness.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Ensure imports + relative paths work regardless of notebook working directory
PROJECT_ROOT = Path.cwd().resolve()
for _ in range(6):
    if (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT / 'models').exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError('Could not locate project root (expected to find ./src and ./models).')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Reload to pick up any recent edits to src/counterfactual.py
import importlib
import src.counterfactual as counterfactual  # noqa: E402
importlib.reload(counterfactual)
generate_counterfactual = counterfactual.generate_counterfactual

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)

## 1) Load a sample patient

We take one row from the saved test split (`data/processed/splits/X_test_raw.csv`)

In [7]:
X_test_raw_path = PROJECT_ROOT / 'data/processed/splits/X_test_raw.csv'
if not X_test_raw_path.exists():
    raise FileNotFoundError('Missing test split. Run: python src/cleaning.py')

X_raw = pd.read_csv(X_test_raw_path)
canonical_cols = ['hemo','sc','al','htn','age','dm']

row_idx = 20  # change this to demo a different patient
patient = X_raw.iloc[row_idx][canonical_cols].to_dict()
patient

{'hemo': 11.3, 'sc': 0.8, 'al': 4.0, 'htn': 0.0, 'age': 7.0, 'dm': 0.0}

## 1b) Optional: auto-pick an “easy” demo patient

Counterfactual search is much faster if the patient is close to the model decision boundary. This cell picks a test patient that is predicted as CKD but already has relatively higher non-CKD probability (typically easiest to flip to non-CKD by improving actionable features).

In [6]:
# Set this to False if you want to stick to row_idx=0 manually
AUTO_PICK_PATIENT = True

if AUTO_PICK_PATIENT:
    model_path = PROJECT_ROOT / 'models/rf_augmented_42_v1.joblib'
    preprocessor_path = PROJECT_ROOT / 'models/preprocessor.pkl'
    model = counterfactual.load_model(model_path)
    preproc = counterfactual.load_preprocessor(preprocessor_path)

    X_candidates = X_raw[canonical_cols].head(50).copy()
    X_canon = preproc.transform(X_candidates)
    proba = model.predict_proba(X_canon)
    pred = model.predict(X_canon)
    classes = getattr(model, 'classes_', None)
    if classes is None or len(classes) != proba.shape[1]:
        raise RuntimeError('Model classes_ missing or inconsistent with predict_proba output.')

    # Prefer picking a CKD-predicted patient and target non-CKD (label 0) if present
    classes_list = [int(x) for x in classes]
    if 0 in classes_list and 1 in classes_list:
        idx_nonckd = classes_list.index(0)
        idx_ckd = classes_list.index(1)
        mask_ckd = (pred.astype(int) == 1)
        if mask_ckd.any():
            # choose CKD cases that already have higher non-CKD probability (closest to boundary)
            cand_idx = (proba[mask_ckd, idx_nonckd]).argmax()
            row_idx = int(X_candidates.index[mask_ckd][cand_idx])
        else:
            # fallback: just choose the most uncertain sample overall
            margin = (proba[:, idx_ckd] - proba[:, idx_nonckd]).astype(float)
            row_idx = int(X_candidates.index[abs(margin).argmin()])
    else:
        # Generic fallback: pick the most uncertain sample (smallest top-2 margin)
        sorted_p = np.sort(proba, axis=1)[:, ::-1]  # descending sort per row
        margin = (sorted_p[:, 0] - sorted_p[:, 1]).astype(float)
        row_idx = int(X_candidates.index[int(np.argmin(margin))])

    patient = X_raw.loc[row_idx, canonical_cols].to_dict()
    print('AUTO_PICK row_idx =', row_idx)
    print('patient =', patient)

AUTO_PICK row_idx = 8
patient = {'hemo': 15.0, 'sc': 1.1, 'al': 1.0, 'htn': 0.0, 'age': 44.0, 'dm': 0.0}


## 2) Generate counterfactuals

Recommended live-demo settings (to flip **CKD → NotCKD**):

- `target_class=0` (explicitly target NotCKD; in this repo `ckd=1`, `notckd=0`)
- `selection='pareto'` (returns Pareto-optimal set)
- `proximity_mode='zscore'` (standardized proximity)
- `max_iter=80`, `target_prob=0.80` (usually fast enough for a demo; adjust if needed)

Note: you *can* set `target_class=None` to auto-flip the model’s current predicted label, but if you accidentally pick a patient that is already NotCKD, auto-flip will target CKD (the opposite of what we want here).

In [8]:
# Paths (explicit, so it works regardless of current working directory)
model_path = PROJECT_ROOT / 'models/rf_augmented_42_v1.joblib'
preprocessor_path = PROJECT_ROOT / 'models/preprocessor.pkl'

if not model_path.exists():
    raise FileNotFoundError(f'Model not found: {model_path}')
if not preprocessor_path.exists():
    raise FileNotFoundError(f'Preprocessor not found: {preprocessor_path}. Run: python src/cleaning.py')

# Sanity-check the original prediction (in this repo: 1=CKD, 0=NotCKD)
_model = counterfactual.load_model(model_path)
_preproc = counterfactual.load_preprocessor(preprocessor_path)
_x0 = counterfactual.prepare_patient_canonical(patient, preprocessor=_preproc, preprocess=True)
print('model classes_:', getattr(_model, 'classes_', None))
print('original predicted label:', int(_model.predict(_x0)[0]))
print('original proba:', counterfactual.predict_proba_cf(_model, _x0)[0])

# Demo settings tuned for speed + high chance of success
weights = counterfactual.LossWeights(lambda_rob=0.0)

cfs_df, metrics_df, explanation, comparison = generate_counterfactual(
    patient,
    model_path=model_path,
    preprocessor_path=preprocessor_path,
    target_class=0,  # NotCKD
    k=2,
    max_iter=60,
    n_neighbors=12,
    pool_size=4,
    max_attempts=25,
    target_prob=0.60,
    proximity_mode='zscore',
    selection='pareto',
    compute_explanation=False,
    explanation_stability_runs=0,
    seed=42,
    weights=weights,
 )

print('auto_target:', explanation.get('auto_target'))
print('resolved target label:', explanation.get('target_class'))
print('resolved target index:', explanation.get('target_index'))

metrics_df

model classes_: [0 1]
original predicted label: 1
original proba: [0.00220238 0.99779762]
auto_target: False
resolved target label: 0
resolved target index: 0


,validity,proximity,sparsity,p_target,robustness


## 2b) Optional: compute SHAP explanation (run only if needed)

The counterfactual search above can be slow if it also computes SHAP. This optional step computes SHAP **after** the counterfactual is selected.

In [5]:
import pandas as pd

if metrics_df.empty:
    print('No valid CF found; skipping SHAP explanation.')
else:
    # Pick the best CF by highest p_target then lowest proximity
    best_row = metrics_df.sort_values(by=['p_target', 'proximity'], ascending=[False, True]).index[0]
    best_cf = cfs_df.loc[[best_row]][['hemo','sc','al','htn','age','dm']].copy()
    x_original = comparison[comparison['row'] == 'original'][['hemo','sc','al','htn','age','dm']].copy()
    # Replace counterfactual row with our selected best_cf (same schema)
    x_cf_best = best_cf.copy()

    model = counterfactual.load_model(model_path)
    target_label = int(explanation.get('target_class'))
    target_index = int(explanation.get('target_index'))

    explanation = counterfactual.explain_original_vs_counterfactual(
        model=model,
        x_original=x_original,
        x_cf_best=x_cf_best,
        target_label=target_label,
        target_index=target_index,
        stability_runs=0,
        rng=None,
    )
    explanation['auto_target'] = True
    print('SHAP available:', explanation.get('shap_available'))

No valid CF found; skipping SHAP explanation.


In [4]:

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

FEATURE_LABELS = {
    'hemo': 'Hemoglobin (g/dL)',
    'sc':   'Serum Creatinine (mg/dL)',
    'al':   'Albumin (g/dL)',
    'htn':  'Hypertension',
    'age':  'Age (years)',
    'dm':   'Diabetes Mellitus',
}
CANONICAL_FEATURES = ['hemo', 'sc', 'al', 'htn', 'age', 'dm']
SHAP_ZERO_THRESH = 1e-4

shap_orig = explanation.get('shap_original')
orig_proba = explanation.get('original_proba', [None, None])
cf_proba   = explanation.get('counterfactual_proba', [None, None])

if not shap_orig:
    print("No SHAP values available. Run the SHAP explanation cell (cell 2b) first.")
else:
    # ── Build sorted items (non-zero first, then zeros) ─────────────────────
    all_items = sorted(
        [(f, shap_orig.get(f, 0.0)) for f in CANONICAL_FEATURES],
        key=lambda x: abs(x[1]),
        reverse=True,
    )
    active = [(f, v) for f, v in all_items if abs(v) > SHAP_ZERO_THRESH]
    zero   = [(f, v) for f, v in all_items if abs(v) <= SHAP_ZERO_THRESH]
    items  = active + zero          # zeros at the bottom

    labels = [FEATURE_LABELS.get(f, f) for f, _ in items]
    values = [float(v) for _, v in items]
    colors = ['#B42318' if v > SHAP_ZERO_THRESH
              else '#1F8F4A' if v < -SHAP_ZERO_THRESH
              else '#CBD5E1'
              for v in values]

    max_abs = max((abs(v) for _, v in active), default=0.1) or 0.1

    # ── Figure ───────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(
        1, 2,
        figsize=(14, max(4.5, 0.75 * len(items) + 1.5)),
        gridspec_kw={'width_ratios': [2.2, 1]},
    )
    fig.suptitle('SHAP Explanation — Selected Patient', fontsize=14, fontweight='bold', y=1.01)

    # ── Left: SHAP waterfall-style bar chart ─────────────────────────────────
    ax = axes[0]
    bars = ax.barh(labels, values, color=colors, edgecolor='white', linewidth=0.6, height=0.58)

    # Value labels
    for bar, val in zip(bars, values):
        if abs(val) > SHAP_ZERO_THRESH:
            ha  = 'left'  if val >= 0 else 'right'
            off = max_abs * 0.04 if val >= 0 else -max_abs * 0.04
            ax.text(val + off, bar.get_y() + bar.get_height() / 2,
                    f'{val:+.4f}', va='center', ha=ha, fontsize=9.5, color='#1E293B')
        else:
            ax.text(0, bar.get_y() + bar.get_height() / 2,
                    '  0  (no contribution)', va='center', ha='left', fontsize=8.5, color='#94A3B8')

    ax.axvline(0, color='#64748B', linewidth=1.4)
    ax.set_xlim(-max_abs * 1.65, max_abs * 1.65)
    ax.set_xlabel('SHAP value  (+  →  higher CKD risk  |  −  →  lower CKD risk)', fontsize=10)
    ax.set_title('Feature contributions for this patient', fontsize=11, pad=8)
    ax.tick_params(axis='y', labelsize=10)
    ax.tick_params(axis='x', labelsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_facecolor('#FAFAFA')

    # Legend
    red_patch   = mpatches.Patch(color='#B42318', label='↑ Increases CKD risk')
    green_patch = mpatches.Patch(color='#1F8F4A', label='↓ Decreases CKD risk')
    grey_patch  = mpatches.Patch(color='#CBD5E1', label='No contribution')
    ax.legend(handles=[red_patch, green_patch, grey_patch],
              loc='lower right', fontsize=8.5, framealpha=0.85)

    # ── Right: Probability comparison ────────────────────────────────────────
    ax2 = axes[1]
    categories = ['Not CKD', 'CKD']
    x = np.arange(len(categories))
    w = 0.32

    def _pct(v): return v * 100 if v is not None else 0

    if None not in orig_proba:
        b1 = ax2.bar(x - w/2, [_pct(orig_proba[0]), _pct(orig_proba[1])],
                     width=w, color=['#1F8F4A', '#B42318'], alpha=0.45,
                     label='Original', edgecolor='white')
        for bar in b1:
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                     f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8.5)

    if cf_proba and None not in cf_proba:
        b2 = ax2.bar(x + w/2, [_pct(cf_proba[0]), _pct(cf_proba[1])],
                     width=w, color=['#1F8F4A', '#B42318'], alpha=1.0,
                     label='Counterfactual', edgecolor='white')
        for bar in b2:
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                     f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8.5)
    else:
        orig_only_note = True

    ax2.set_xticks(x)
    ax2.set_xticklabels(categories, fontsize=10)
    ax2.set_ylabel('Probability (%)', fontsize=10)
    ax2.set_ylim(0, 115)
    ax2.set_title('Prediction probability', fontsize=11, pad=8)
    ax2.legend(fontsize=8.5)
    ax2.spines[['top', 'right']].set_visible(False)
    ax2.set_facecolor('#FAFAFA')
    ax2.tick_params(axis='y', labelsize=9)

    plt.tight_layout()
    plt.show()

    # ── Text summary ─────────────────────────────────────────────────────────
    print('\n── Top contributing features ──')
    for f, v in active:
        direction = 'increases' if v > 0 else 'decreases'
        print(f'  {FEATURE_LABELS[f]:<28}  SHAP = {v:+.4f}  →  {direction} CKD risk')
    if zero:
        names = ', '.join(FEATURE_LABELS.get(f, f) for f, _ in zero)
        print(f'\n  No contribution: {names}')


No SHAP values available. Run the SHAP explanation cell (cell 2b) first.


## 3) View the counterfactual(s) and comparison table

In [5]:
display(cfs_df)

comparison

,hemo,sc,al,htn,age,dm
0,14.497566,0.4,0.044841,0.0,14.0,0.0
1,14.492545,0.8,0.000000,0.0,14.0,0.0


,row,hemo,sc,al,htn,age,dm
0,original,14.300000,0.8,0.000000,0.0,14.0,1.0
1,best_counterfactual,14.497566,0.4,0.044841,0.0,14.0,0.0


## 4) Explanation summary

- `clinical_text` is a human-readable change summary.
- If SHAP is installed and compatible with the model, you will also see SHAP deltas and a top-k overlap metric.

In [6]:
print(explanation.get('clinical_text', explanation.get('note', 'No clinical text available.')))

# Show key explanation fields (if SHAP ran)
keys_to_show = [
    'original_proba', 'counterfactual_proba',
    'top_features_changed',
    'top_shap_delta_features',
    'shap_topk_overlap',
    'shap_input_stability',
]
{k: explanation.get(k) for k in keys_to_show if k in explanation}

Clinical interpretation (feature changes):
- serum creatinine (sc) decreased from 0.80 to 0.40
- hemoglobin (hemo) increased from 14.30 to 14.50
- albumin (al) increased from 0.00 to 0.04
- diabetes (dm) changed from 1 to 0


{'original_proba': [0.22519241486831748, 0.7748075851316826],
 'counterfactual_proba': [0.9624943214937978, 0.037505678506202525],
 'top_features_changed': ['hemo', 'sc', 'al', 'dm']}

## Notes / troubleshooting

1. If you get `No valid counterfactual found...`, try: 
   - lowering `target_prob` (e.g., 0.75), or
   - increasing `max_iter` (e.g., 150–300), or
   - increasing `k` and letting Pareto selection pick good candidates.

2. If SHAP is missing/unsupported, the pipeline still works; the explanation dict will include a note.

3. The search is heuristic and may converge to local optima; the implementation uses restarts to reduce that risk.